In [ ]:
import os
import cv2
import random
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.models as models
from torchvision import datasets, transforms
from sklearn.metrics import cohen_kappa_score
from PIL import Image
import matplotlib.pyplot as plt


DATASET_BASE = '/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2'
TRAIN_DIR = os.path.join(DATASET_BASE, 'train')
VAL_DIR = os.path.join(DATASET_BASE, 'val')
IMAGE_SIZE = 224       
BATCH_SIZE = 32       
EPOCHS = 5           
LR = 1e-4             
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. PREPROCESSING FUNCTIONS

In [ ]:

def crop_and_enhance(pil_img):

    img = np.array(pil_img)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        img = img[y:y+h, x:x+w]
        
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), 10), -4, 128)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    return Image.fromarray(img)


train_transform = transforms.Compose([
    transforms.Lambda(crop_and_enhance),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


val_transform = transforms.Compose([
    transforms.Lambda(crop_and_enhance),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



# 3. VISUALIZATION FUNCTION


In [ ]:

def visualize_preprocessing():
    print("\n--- Generating Preprocessing Visualization ---")
    severe_folder = os.path.join(TRAIN_DIR, '4')
    if not os.path.exists(severe_folder):
        print(f"Folder not found: {severe_folder}")
        return
        
    images = [f for f in os.listdir(severe_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if not images:
        return
        
    sample_path = os.path.join(severe_folder, random.choice(images))
    raw_pil = Image.open(sample_path).convert('RGB')
    enhanced_pil = crop_and_enhance(raw_pil)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(raw_pil)
    axes[0].set_title("Before: Raw Fundus Image", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(enhanced_pil)
    axes[1].set_title("After: Cropped Enhanced", fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()



# 4. MODEL CREATION FUNCTION


In [ ]:

def build_model():
    
    model = models.densenet121(weights="DEFAULT")
    in_features = model.classifier.in_features
    
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 1) 
    )
    return model

# 5. TRAINING ENGINE FUNCTIONS


In [ ]:



def run_one_epoch(model, dataloader, optimizer, criterion, scaler, is_train=True):
    if is_train:
        model.train()
    else:
        model.eval()
        
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.set_grad_enabled(is_train):
        for images, labels in dataloader:
            images = images.to(DEVICE)
           
            labels = labels.to(DEVICE).float() 
            
            if is_train:
                optimizer.zero_grad()
            
            with autocast():
                predictions = model(images)
                loss = criterion(predictions.squeeze(-1), labels)
                
            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
            total_loss += loss.item()
            
            preds_np = predictions.detach().cpu().squeeze(-1).numpy()
            labels_np = labels.detach().cpu().numpy()
            
            all_preds.extend(preds_np)
            all_labels.extend(labels_np)
            
    avg_loss = total_loss / len(dataloader)
    
    # Calculate QWK 
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    preds_rounded = np.clip(np.round(all_preds), 0, 4).astype(int)
    labels_int = all_labels.astype(int)
    qwk = cohen_kappa_score(labels_int, preds_rounded, weights='quadratic')
    
    return avg_loss, qwk

In [ ]:
def main():
    #Trigger the visualization before starting the heavy lifting
    visualize_preprocessing()
    
    print("\nPreparing Datasets via ImageFolder...")
    
    train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
    val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_transform)
    
    print(f"Training Images: {len(train_dataset)}")
    print(f"Validation Images: {len(val_dataset)}")
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    model = build_model().to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.MSELoss() 
    scaler = GradScaler()
    
    best_val_qwk = -1.0 
    
    print("\n--- Starting DenseNet121 Training ---")
    for epoch in range(EPOCHS):
        train_loss, train_qwk = run_one_epoch(model, train_loader, optimizer, criterion, scaler, is_train=True)
        val_loss, val_qwk = run_one_epoch(model, val_loader, None, criterion, scaler, is_train=False)
        
        print(f"Epoch [{epoch+1}/{EPOCHS}]")
        print(f"  Train - Loss: {train_loss:.4f} | QWK: {train_qwk:.4f}")
        print(f"  Val - Loss: {val_loss:.4f} | QWK: {val_qwk:.4f}")
        
        if val_qwk > best_val_qwk:
            best_val_qwk = val_qwk
            best_save_path = f"/kaggle/working/best_densenet_qwk.pth"
            torch.save(model.state_dict(), best_save_path)
            print(f"Saved weights to {best_save_path}")

if __name__ == '__main__':
    main()